In [ ]:
import requests
import pandas as pd
import numpy as np 
import json

# 데이터수집

# 년도별 총 헌혈자수


In [ ]:
year = pd.read_csv('./데이터원본/year(2005~2024).csv')
year.head()
year.info()
year.describe()

year.to_csv("year_clean.csv", index=False, encoding="utf-8-sig")

In [ ]:
year = pd.read_csv('./전처리/year_clean.csv')
year.columns
# 컬럼정리
year = year[['시점','헌혈가능인구 (16~69세) (명)','헌혈자 실인원 (명)','헌혈자 1인당 평균 헌혈실적 (건)','총 헌혈실적 (건)']]

# 컬럼이름변경
year = year.rename(columns={
    '헌혈가능인구 (16~69세) (명)': '헌혈가능인구',
    '헌혈자 실인원 (명)': '헌혈자수',
    '헌혈자 1인당 평균 헌혈실적 (건)': '1인당평균헌혈',
    '총 헌혈실적 (건)': '총헌혈건수',
    '시점':'년도'
})

year.info()
year.tail()

#파일저장
year.to_csv("year2.csv", index=False, encoding="utf-8-sig")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   년도       20 non-null     int64  
 1   헌혈가능인구   20 non-null     int64  
 2   헌혈자수     20 non-null     int64  
 3   1인당평균헌혈  20 non-null     float64
 4   총헌혈건수    20 non-null     int64  
dtypes: float64(1), int64(4)
memory usage: 932.0 bytes


# 월별 헌혈건수

In [ ]:
url = 'https://kosis.kr/openapi/Param/statisticsParameterData.do?method=getList&apiKey=NDM2YzQ2NTdiNTBkM2QyNDg4ZjYyNDYzNzg4OTQwY2U=&itmId=T001+T002+T003+T004+T005+T006+T007+T008+&objL1=&objL2=&objL3=&objL4=&objL5=&objL6=&objL7=&objL8=&format=json&jsonVD=Y&prdSe=Y&startPrdDe=2005&endPrdDe=2024&orgId=445&tblId=DT_445001_007'
response = requests.get(url)

In [75]:
# api 함수생성

import requests
import pandas as pd

def get_data(url):
    response = requests.get(url)
    data = response.json()
    
    df = pd.DataFrame(data)
    return df
month = get_data(url)
# month.head()
month.head()

#원본저장
month.to_csv("month(2005~2024).csv", index=False, encoding="utf-8-sig")

In [ ]:
# 한글변경
month= month.rename(columns={
    'C1_OBJ_NM': '성별',
    'C2_NM': '월',
    'DT': '헌혈건수',
    'PRD_DE': '연도',
    'C1_NM': '구분',
    'TBL_NM': '테이블명',
    'UNIT_NM':'단위'
})

month=month[['헌혈건수','월','단위','연도']]



In [ ]:
# 데이터확인
month.head()

# # 월컬럼 합계 행 삭제
month=month[month['월'] != '합계']
month['월']

# 단위 컬럼 %행 삭제
month = month[month['단위'] != '%']

# # 단위 컬럼삭제
month = month.drop(columns=['단위'])

# # 데이터 타입확인
# month.info()

# 결측치 - -> 0
month['헌혈건수'] = month['헌혈건수'].fillna(0)

# 전체 컬럼 공백 제거 
month.columns = month.columns.str.strip()

# 타입변환
month['헌혈건수'] = pd.to_numeric(month['헌혈건수'], errors='coerce')
month['연도'] = pd.to_numeric(month['연도'], errors='coerce')
month['월'] = (
    month['월']
    .astype(str)
    .str.replace('월', '')
    .pipe(pd.to_numeric, errors='coerce')
)

# 년도,월 컬럼 정렬
month = month.sort_values(['연도', '월']).reset_index(drop=True)
month.head(5)

# month['헌혈건수']

0        169087.0
1        165863.0
2         15577.0
3         10614.0
4         13714.0
5         14671.0
6         16351.0
7         13780.0
8          9870.0
9          5617.0
10        11535.0
11         6816.0
12         4899.0
13        13057.0
14         7654.0
15        12237.0
16         7125.0
17         2346.0
18         3224.0
19       141112.0
20       138540.0
21        12876.0
22         9645.0
23        11317.0
24        12148.0
25        12258.0
26        11495.0
27         7304.0
28         4618.0
29         9798.0
30         6459.0
31         4200.0
32        11922.0
33         6639.0
34         9603.0
35         6526.0
36         1732.0
37         2572.0
38        27975.0
39        27323.0
40         2701.0
41          969.0
42         2397.0
43         2523.0
44         4093.0
45         2285.0
46         2566.0
47          999.0
48         1737.0
49          357.0
50          699.0
51         1135.0
52         1015.0
53         2634.0
54          599.0
55        

In [ ]:
# 결측치확인
print(month['월'].unique())
print(month['월'].isna().sum())

# month[['연도', '월']].head(20)

# 데이터 연도,월로 groupby
month = month.groupby(['연도', '월'])['헌혈건수'].sum().reset_index()
month.head()

month['연도'].unique()
#[2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015,
#        2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024])

[ 1  2  3  4  5  6  7  8  9 10 11 12]
0


array([2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015,
       2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024])

In [ ]:
# 파일저장
month.to_csv("month_clean.csv", index=False, encoding="utf-8-sig")

#전처리파일저장
month = pd.read_csv('./전처리/month_clean.csv')
month.head()
month.info()
month.to_csv('전처리2/month2.csv', index=False, encoding='utf-8-sig')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   연도      240 non-null    int64  
 1   월       240 non-null    int64  
 2   헌혈건수    240 non-null    float64
dtypes: float64(1), int64(2)
memory usage: 5.8 KB


# 총헌혈수 (지역별)

In [ ]:
url ='https://kosis.kr/openapi/Param/statisticsParameterData.do?method=getList&apiKey=NDM2YzQ2NTdiNTBkM2QyNDg4ZjYyNDYzNzg4OTQwY2U=&itmId=T001+T002+T003+&objL1=ALL&objL2=&objL3=&objL4=&objL5=&objL6=&objL7=&objL8=&format=json&jsonVD=Y&prdSe=Y&startPrdDe=2005&endPrdDe=2024&orgId=445&tblId=DT_445001_013'

import requests
import pandas as pd

def get_data(url):
    response = requests.get(url)
    data = response.json()
    
    df = pd.DataFrame(data)
    return df
region = get_data(url)


region.head() # C1_NM 합계,지역이름

#원본저장
# region.to_csv("region(2025~2024).csv", index=False, encoding="utf-8-sig")

,C1_OBJ_NM,DT,C1,PRD_SE,UNIT_NM_ENG,ITM_ID,TBL_ID,ITM_NM,TBL_NM,PRD_DE,LST_CHN_DE,C1_NM,UNIT_NM,ORG_ID
0,시·도별,2274336,A00,A,Case,T001,DT_445001_013,헌혈실적,시·도별 인구대비 헌혈실적,2005,2018-07-02,합계,건,445
1,시·도별,2302542,A00,A,Case,T001,DT_445001_013,헌혈실적,시·도별 인구대비 헌혈실적,2006,2018-07-02,합계,건,445
2,시·도별,2087762,A00,A,Case,T001,DT_445001_013,헌혈실적,시·도별 인구대비 헌혈실적,2007,2018-07-02,합계,건,445
3,시·도별,2347184,A00,A,Case,T001,DT_445001_013,헌혈실적,시·도별 인구대비 헌혈실적,2008,2018-07-02,합계,건,445
4,시·도별,2569954,A00,A,Case,T001,DT_445001_013,헌혈실적,시·도별 인구대비 헌혈실적,2009,2018-07-02,합계,건,445


In [ ]:
# 한글컬럼변경
region = region.rename(columns={
    "C1_OBJ_NM": "시도구분",
    "DT": "값",
    "C1": "시도코드",
    "PRD_SE": "기간구분",
    "UNIT_NM_ENG": "단위(영문)",
    "ITM_ID": "항목ID",
    "TBL_ID": "테이블ID",
    "ITM_NM": "항목명",
    "TBL_NM": "테이블명",
    "PRD_DE": "기준연도",
    "LST_CHN_DE": "최종수정일",
    "C1_NM": "시도명",
    "UNIT_NM": "단위",
    "ORG_ID": "기관ID"
})

region.head()


# 필요한 컬럼만 남기기

region = region[["시도명", "기준연도", "값", "항목명",'단위']]
region.head()

print(region.columns.tolist())



['시도명', '기준연도', '값', '항목명', '단위']


In [ ]:
print(region[region['단위']=='명']['값'].head(20))

40    48138077
41    48297184
42    48456369
43    48606787
44    48746693
45    48874539
46    48988833
47    50004441
48    50219669
49    50423955
50    50617045
51    50801405
52    51446201
53    51635256
54    51849861
55    51829023
56    51638809
57    51439038
58    51325329
59    51217221
Name: 값, dtype: object


In [ ]:
region.info()

# 값 전처리 (문자 → 숫자)
region['값'] = (
    region['값']
    .astype(str)
    .str.replace(',', '')   # 쉼표 제거
    .str.strip()            # 공백 제거
)

# 2숫자 변환
region['값'] = pd.to_numeric(region['값'], errors='coerce')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 840 entries, 0 to 839
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   시도명     840 non-null    object
 1   기준연도    840 non-null    object
 2   값       840 non-null    object
 3   항목명     840 non-null    object
 4   단위      840 non-null    object
dtypes: object(5)
memory usage: 32.9+ KB


In [ ]:
#  합계 제거
region = region[region['시도명'] != '합계']

# 컬럼 삭제
# region = region.drop(columns=['항목명'])

# 단위컬럼 정리
region = region[region['단위'] != '%']


In [ ]:
# 건,명 컬럼생성
# region['값'] = region.to_numeric(df['값'], errors='coerce')

region = region.pivot_table(
    index=['시도명', '기준연도'],
    columns='단위',
    values='값',
    aggfunc='first'
).reset_index()

# 컬럼 이름 정리
region_pivot.columns.name = None



In [ ]:
print(region.columns)
region.head()

# 건,명 결측치확인
region['건'].isnull().sum()
region['명'].isnull().sum()


Index(['시도명', '기준연도', '건', '명'], dtype='object', name='단위')


np.int64(0)

In [ ]:
# 데이터확인
region.info()
# region

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 260 entries, 0 to 259
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   시도명     260 non-null    object 
 1   기준연도    260 non-null    object 
 2   건       260 non-null    float64
 3   명       260 non-null    float64
dtypes: float64(2), object(2)
memory usage: 8.3+ KB


In [ ]:
region.to_csv("region_clean.csv", index=False, encoding="utf-8-sig")

# 연령별 헌혈 통계

In [ ]:

url='https://kosis.kr/openapi/Param/statisticsParameterData.do?method=getList&apiKey=NDM2YzQ2NTdiNTBkM2QyNDg4ZjYyNDYzNzg4OTQwY2U=&itmId=T001+T002+&objL1=ALL&objL2=ALL&objL3=ALL&objL4=&objL5=&objL6=&objL7=&objL8=&format=json&jsonVD=Y&prdSe=Y&startPrdDe=2005&endPrdDe=2024&orgId=445&tblId=DT_445001_003'
def get_data(url):
    response = requests.get(url)
    response.raise_for_status()
    
    data = response.json()
    
    if isinstance(data, dict):
        data = data.get("list", data)
    
    return pd.DataFrame(data)

age = get_data(url)
age.head()

#원본저장
age.to_csv("age(2005~2024).csv", index=False, encoding="utf-8-sig")

rename_dict = {
    "C1_OBJ_NM": "성별구분",
    "C1_OBJ_NM_ENG": "성별구분(영문)",

    "C1": "성별코드",
    "C1_NM": "성별명",
    "C1_NM_ENG": "성별명(영문)",

    "C2_OBJ_NM": "연령구분",
    "C2": "연령코드",
    "C2_NM": "연령명",

    "C3_OBJ_NM": "혈액원구분",
    "C3": "혈액원코드",
    "C3_NM": "혈액원명",

    "DT": "헌혈건수",

    "PRD_DE": "기준연도",
    "PRD_SE": "기간구분",

    "ITM_ID": "항목ID",
    "ITM_NM": "항목명",

    "TBL_ID": "테이블ID",
    "TBL_NM": "테이블명",

    "UNIT_NM": "단위",
    "UNIT_NM_ENG": "단위(영문)",

    "LST_CHN_DE": "수정일",
    "ORG_ID": "기관ID"
}

age= age.rename(columns=rename_dict)
age.head()

# age['연령명'].unique() #'합계', '16~19세', '20~29세', '30~39세', '40~49세', '50~59세', '60세이상'
age.columns


Index(['성별구분', '연령명', '혈액원코드', '헌혈건수', '연령코드', '성별코드', '기간구분', '단위(영문)',
       '항목ID', '테이블ID', '항목명', '테이블명', '기준연도', '수정일', '성별명', '단위', '혈액원명',
       '혈액원구분', '기관ID', '성별구분(영문)', '연령구분', '성별명(영문)'],
      dtype='object')

In [ ]:
age = age[[
    "연령코드",
    "연령명",
    "기준연도",
    "헌혈건수",
    '단위'
]]

# A001 제거
age = age[age["연령코드"] != "A001"]

# 코드 변환 
age_map = {
    "A002": 10,
    "A003": 20,
    "A004": 30,
    "A005": 40,
    "A006": 50,
    "A007": 60
}

age["연령코드"] = age["연령코드"].map(age_map)

# 타입 안정화
age = age.dropna(subset=['연령코드'])
age["연령코드"] = age["연령코드"].astype(int)

# 연령대 라벨 생성
age_label = {
    10: "16~19세",
    20: "20~29세",
    30: "30~39세",
    40: "40~49세",
    50: "50~59세",
    60: "60세이상"
}

age["연령대"] = age["연령코드"].map(age_label)

# 확인
# age

# 헌혈건수 분리
age['단위'].unique()



array(['건', '%'], dtype=object)

In [ ]:
age.columns

Index(['연령코드', '연령명', '기준연도', '헌혈건수', '단위', '연령대'], dtype='object')

In [ ]:
age.head()

age = age.pivot_table(
    index=["연령코드", "연령대", "기준연도",'연령명'],
    columns="단위",
    values="헌혈건수",
    aggfunc="first"
).reset_index()

age.head()

단위,연령코드,연령대,기준연도,연령명,%,건
0,10,16~19세,2005,16~19세,31.1,707262
1,10,16~19세,2006,16~19세,33,760145
2,10,16~19세,2007,16~19세,35.2,735107
3,10,16~19세,2008,16~19세,36.3,851751
4,10,16~19세,2009,16~19세,34.9,897580


In [ ]:

# 컬럼삭제
# age =age.drop(columns=['연령명','%'])
# age.info()

age['기준연도'] = age['기준연도'].astype(int)
age['건'] = age['건'].astype(int)
# 데이터저장

age.to_csv("age_clean.csv", index=False, encoding="utf-8-sig")

# 직업별 통계


In [ ]:

url='https://kosis.kr/openapi/Param/statisticsParameterData.do?method=getList&apiKey=NDM2YzQ2NTdiNTBkM2QyNDg4ZjYyNDYzNzg4OTQwY2U=&itmId=T001+T002+&objL1=ALL&objL2=ALL&objL3=ALL&objL4=&objL5=&objL6=&objL7=&objL8=&format=json&jsonVD=Y&prdSe=Y&startPrdDe=2005&endPrdDe=2024&orgId=445&tblId=DT_445001_004'
def get_data(url):
    response = requests.get(url)
    response.raise_for_status()
    
    data = response.json()
    
    if isinstance(data, dict):
        data = data.get("list", data)
    
    return pd.DataFrame(data)

job = get_data(url)
job.head()
# job.columns

#원본저장
# job.to_csv("job(2005~2024).csv", index=False, encoding="utf-8-sig")

rename_dict = {
    "C1_OBJ_NM": "성별구분",
    "C1_OBJ_NM_ENG": "성별구분(영문)",

    "C1": "성별코드",
    "C1_NM": "성별명",
    "C1_NM_ENG": "성별명(영문)",

    "C2_OBJ_NM": "직업구분",
    "C2": "직업코드",
    "C2_NM": "직업명",

    "C3_OBJ_NM": "혈액원구분",
    "C3": "혈액원코드",
    "C3_NM": "혈액원명",

    "DT": "헌혈건수",

    "PRD_DE": "기준연도",
    "PRD_SE": "기간구분",

    "ITM_ID": "항목ID",
    "ITM_NM": "항목명",

    "TBL_ID": "테이블ID",
    "TBL_NM": "테이블명",

    "UNIT_NM": "단위",
    "UNIT_NM_ENG": "단위(영문)",

    "ORG_ID": "기관ID",
    "LST_CHN_DE": "수정일"
}

job = job.rename(columns=rename_dict)
job.head()
# job['직업명'].unique() # '고등학생', '대학생', '군인', '회사원', '공무원', '자영업', '종교직', '가사', '기타


,성별구분,직업명,혈액원코드,헌혈건수,직업코드,성별코드,기간구분,단위(영문),항목ID,테이블ID,항목명,테이블명,기준연도,수정일,성별명,단위,혈액원명,혈액원구분,기관ID,성별구분(영문),직업구분,성별명(영문)
0,성별,합계,C01,2274336,A01,0,A,Case,T001,DT_445001_004,실적,직업별 헌혈통계,2005,2018-03-23,계,건,합계,혈액원별,445,By gender,직업별,NaN
1,성별,합계,C01,2302542,A01,0,A,Case,T001,DT_445001_004,실적,직업별 헌혈통계,2006,2018-03-23,계,건,합계,혈액원별,445,By gender,직업별,NaN
2,성별,합계,C01,2087762,A01,0,A,Case,T001,DT_445001_004,실적,직업별 헌혈통계,2007,2018-03-23,계,건,합계,혈액원별,445,By gender,직업별,NaN
3,성별,합계,C01,2347184,A01,0,A,Case,T001,DT_445001_004,실적,직업별 헌혈통계,2008,2018-03-23,계,건,합계,혈액원별,445,By gender,직업별,NaN
4,성별,합계,C01,2569954,A01,0,A,Case,T001,DT_445001_004,실적,직업별 헌혈통계,2009,2018-03-23,계,건,합계,혈액원별,445,By gender,직업별,NaN


In [ ]:
# 컬럼정리
job = job[[
    "직업명",
    "직업코드",
    "기준연도",
    "헌혈건수",
    '단위'
]]



# 직업명
job['직업명'].unique()
job = job[~job['직업명'].isin(['합계','기타'])]


# job['직업코드'].unique()
# job = job[~job["혈액원명"].isin(['합계','대한적십자사 외'])]

# 합계삭제,분리

job['헌혈건수']

job = job.pivot_table(
    index=["직업명", "직업코드",'기준연도'],
    columns="단위",
    values="헌혈건수",
    aggfunc="first"
).reset_index()

# %에 0인값이많아서 삭제
job = job.drop(columns=['%'])


In [ ]:
job.head()
# 성별코드,혈액원명 삭제
# job = job.drop(columns=['성별코드','혈액원명'])
job.head()


단위,직업명,직업코드,기준연도,건
0,가사,A09,2005,20529
1,가사,A09,2006,22893
2,가사,A09,2007,20961
3,가사,A09,2008,23988
4,가사,A09,2009,27420


In [ ]:
job.tail()

단위,직업명,직업코드,기준연도,건
155,회사원,A05,2020,849493
156,회사원,A05,2021,847679
157,회사원,A05,2022,914277
158,회사원,A05,2023,956681
159,회사원,A05,2024,980370


In [ ]:
# job['건'].isnull().sum() # 건
# job['건'] = job['건'].fillna(0).astype(int)

job.head(23)
# 건 타입변경
import numpy as np
import pandas as pd

job['건'] = job['건'].replace('-', np.nan)
  
job['건'] = job['건'].astype(int)

# 데이터저장

job.to_csv("job_clean.csv", index=False, encoding="utf-8-sig")

# 장소별통계

In [ ]:
url='https://kosis.kr/openapi/Param/statisticsParameterData.do?method=getList&apiKey=NDM2YzQ2NTdiNTBkM2QyNDg4ZjYyNDYzNzg4OTQwY2U=&itmId=T001+T002+&objL1=ALL&objL2=ALL&objL3=ALL&objL4=&objL5=&objL6=&objL7=&objL8=&format=json&jsonVD=Y&prdSe=Y&startPrdDe=2005&endPrdDe=2024&orgId=445&tblId=DT_445001_005'
def get_data(url):
    response = requests.get(url)
    response.raise_for_status()
    
    data = response.json()
    
    if isinstance(data, dict):
        data = data.get("list", data)
    
    return pd.DataFrame(data)

location = get_data(url)
location.head()

#원본저장
location.to_csv("location(2005~2024).csv", index=False, encoding="utf-8-sig")


rename_dict = {
    "C1_OBJ_NM": "성별구분",
    "C1_OBJ_NM_ENG": "성별구분(영문)",

    "C1": "성별코드",
    "C1_NM": "성별명",
    "C1_NM_ENG": "성별명(영문)",

    "C2_OBJ_NM": "장소구분",
    "C2": "장소코드",
    "C2_NM": "장소명",

    "C3_OBJ_NM": "혈액원구분",
    "C3": "혈액원코드",
    "C3_NM": "혈액원명",

    "DT": "헌혈건수",

    "PRD_DE": "기준연도",
    "PRD_SE": "기간구분",

    "ITM_ID": "항목ID",
    "ITM_NM": "항목명",

    "TBL_ID": "테이블ID",
    "TBL_NM": "테이블명",

    "UNIT_NM": "단위",
    "UNIT_NM_ENG": "단위(영문)",

    "ORG_ID": "기관ID",
    "LST_CHN_DE": "수정일"
}

location= location.rename(columns=rename_dict)
location['장소명'].unique() # 합계', '단체헌혈', '고등학교', '대학교', '군부대', '종교', '일반단체', '개인헌혈', '원내',
                            #'헌혈의집', '가두

                            #가두 : 야외·이동 장소에서 진행하는 헌혈





array(['합계', '단체헌혈', '고등학교', '대학교', '군부대', '종교', '일반단체', '개인헌혈', '원내',
       '헌혈의집', '가두'], dtype=object)

In [ ]:
# 필요한 컬럼
location = location[['장소코드', '장소명', '혈액원명','헌혈건수','단위']]
# 합계 행 삭제
location = location[location['혈액원명'] != '합계']
# 대한적십자사 외 행 확인
len([location['혈액원명'] != '대한적십자사 외']) 
# 1건 -> 삭제
location=location[location['혈액원명'] != '대한적십자사 외']
# 장소명 합계삭제
location = location[location['장소명'] != '합계']


# 장소명에 기관,유형,장소가 섞여있음 분리작업 
len(location['혈액원명']=='대한적십자사')

# 기관구분 컬럼 생성
location['기관구분'] = '지역혈액원'

location.loc[location['혈액원명'] == '대한적십자사', '기관구분'] = '중앙기관'

location.loc[location['혈액원명'] == '대한적십자사 외', '기관구분'] = '기타'
location.head(10)


,장소코드,장소명,혈액원명,헌혈건수,단위,기관구분
406,A01,단체헌혈,대한적십자사,1209466,건,중앙기관
407,A01,단체헌혈,대한적십자사,1060939,건,중앙기관
408,A01,단체헌혈,대한적십자사,899402,건,중앙기관
409,A01,단체헌혈,대한적십자사,965846,건,중앙기관
410,A01,단체헌혈,대한적십자사,923908,건,중앙기관
411,A01,단체헌혈,대한적십자사,883772,건,중앙기관
412,A01,단체헌혈,대한적십자사,827700,건,중앙기관
413,A01,단체헌혈,대한적십자사,880801,건,중앙기관
414,A01,단체헌혈,대한적십자사,922787,건,중앙기관
415,A01,단체헌혈,대한적십자사,928716,건,중앙기관


In [ ]:
location.info()


# 장소명
(location['장소명'] == '종교').sum()
# 일반헌혈 510,단체 510,개인 510

# 헌혈유형 분리하기 (추정)
location['추정헌혈유형'] = pd.Series(index=location.index, dtype='object')

location.loc[
    location['장소명'].isin(['고등학교','대학교','군부대','종교','일반단체','단체헌혈']),
    '추정헌혈유형'
] = '단체'

location.loc[
    location['장소명'].isin(['헌혈의집','가두','원내','개인헌혈']),
    '추정헌혈유형'
] = '개인'

# 장소명 유형 이름 변경

location = location[~location['장소명'].isin(['단체헌혈','개인헌혈','일반단체'])]
location.head()

<class 'pandas.core.frame.DataFrame'>
Index: 19560 entries, 406 to 23017
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   장소코드    19560 non-null  object
 1   장소명     19560 non-null  object
 2   혈액원명    19560 non-null  object
 3   헌혈건수    19560 non-null  object
 4   단위      19560 non-null  object
 5   기관구분    19560 non-null  object
dtypes: object(6)
memory usage: 1.0+ MB


,장소코드,장소명,혈액원명,헌혈건수,단위,기관구분,추정헌혈유형
1138,A0101,고등학교,대한적십자사,293838,건,중앙기관,단체
1139,A0101,고등학교,대한적십자사,321625,건,중앙기관,단체
1140,A0101,고등학교,대한적십자사,308519,건,중앙기관,단체
1141,A0101,고등학교,대한적십자사,337228,건,중앙기관,단체
1142,A0101,고등학교,대한적십자사,284924,건,중앙기관,단체


In [ ]:
# 결측치 -> nan 변경
location['헌혈건수'] = location['헌혈건수'].replace('-', np.nan)
location['헌혈건수'].isnull().sum() # 45개


# 컬럼정리
# location = location.drop(columns=['단위','기관구분'])
location.isnull().sum()
# location[['장소명','헌혈건수']]


장소코드        0
장소명         0
혈액원명        0
헌혈건수      192
추정헌혈유형      0
dtype: int64

In [ ]:
# location['단위'].unique()
location = location[location['단위'] != '%']
location.head()



,장소코드,장소명,혈액원명,헌혈건수,단위,기관구분,추정헌혈유형
1138,A0101,고등학교,대한적십자사,293838,건,중앙기관,단체
1139,A0101,고등학교,대한적십자사,321625,건,중앙기관,단체
1140,A0101,고등학교,대한적십자사,308519,건,중앙기관,단체
1141,A0101,고등학교,대한적십자사,337228,건,중앙기관,단체
1142,A0101,고등학교,대한적십자사,284924,건,중앙기관,단체


In [ ]:
# 결측치 제거
# 가두,지역혈액원,A0203 매번다른장소에서 진행되는 가두는 헌혈건수측정 xx
# 0으로 대체
location[location['헌혈건수'].isna()]
location['헌혈건수'] = location['헌혈건수'].fillna(0)
location['헌혈건수'] = location['헌혈건수'].astype(int)


location.isnull().sum()
location.info()
# 데이터저장
location.to_csv("location_clean.csv", index=False, encoding="utf-8-sig")

# location.isnull().sum()


<class 'pandas.core.frame.DataFrame'>
Index: 6846 entries, 1138 to 22997
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   장소코드    6846 non-null   object
 1   장소명     6846 non-null   object
 2   혈액원명    6846 non-null   object
 3   헌혈건수    6846 non-null   int64 
 4   추정헌혈유형  6846 non-null   object
dtypes: int64(1), object(4)
memory usage: 320.9+ KB


# 혈액형별 통계

In [ ]:
url='https://kosis.kr/openapi/Param/statisticsParameterData.do?method=getList&apiKey=NDM2YzQ2NTdiNTBkM2QyNDg4ZjYyNDYzNzg4OTQwY2U=&itmId=T001+T002+&objL1=ALL&objL2=ALL&objL3=ALL&objL4=&objL5=&objL6=&objL7=&objL8=&format=json&jsonVD=Y&prdSe=Y&startPrdDe=2005&endPrdDe=2024&orgId=445&tblId=DT_445001_006'

def get_data(url):
    response = requests.get(url)
    response.raise_for_status()
    
    data = response.json()
    
    if isinstance(data, dict):
        data = data.get("list", data)
    
    return pd.DataFrame(data)

Bloodtype = get_data(url)
Bloodtype.head()

#원본저장
Bloodtype.to_csv("Bloodtype(2005~2024).csv", index=False, encoding="utf-8-sig")

rename_dict = {

    "C1_OBJ_NM": "성별구분",
    "C1_OBJ_NM_ENG": "성별구분(영문)",

    "C1": "성별코드",
    "C1_NM": "성별명",
    "C1_NM_ENG": "성별명(영문)",

    "C2_OBJ_NM": "혈액형구분",
    "C2": "혈액형코드",
    "C2_NM": "혈액형명",

    "C3_OBJ_NM": "혈액원구분",
    "C3": "혈액원코드",
    "C3_NM": "혈액원명",

    "DT": "헌혈건수",

    "PRD_DE": "기준연도",
    "PRD_SE": "기간구분",

    "ITM_ID": "항목ID",
    "ITM_NM": "항목명",

    "TBL_ID": "테이블ID",
    "TBL_NM": "테이블명",

    "UNIT_NM": "단위",
    "UNIT_NM_ENG": "단위(영문)",

    "ORG_ID": "기관ID",
    "LST_CHN_DE": "수정일"
}

# 컬럼명 변경
Bloodtype = Bloodtype.rename(columns=rename_dict)

In [ ]:
# 컬럼 삭제
# 필요한 컬럼만 선택
Bloodtype = Bloodtype[['헌혈건수', '혈액형명', '혈액형코드','단위']]
Bloodtype.head()

,헌혈건수,혈액형명,혈액형코드,단위
0,2274336,합계,A00,건
1,2302542,합계,A00,건
2,2087762,합계,A00,건
3,2347184,합계,A00,건
4,2569954,합계,A00,건


In [ ]:
# 혈액형명

# 혈액형명 합계삭제
# Bloodtype = Bloodtype[~(Bloodtype['혈액형명']=='합계')]
# 단위 % 삭제
Bloodtype = Bloodtype[~(Bloodtype['단위']=='%')]

# ~형 삭제
Bloodtype['혈액형명'].unique() #['O형', 'A형', 'B형', 'AB형']
Bloodtype['혈액형명'] = Bloodtype['혈액형명'].str.replace('형','',regex=False)
# 혈액형코드 A00삭제
Bloodtype = Bloodtype[~(Bloodtype['혈액형코드']=='A01')]

#수혈의 안정성때문에 분리되어있음
# RH+ 
Bloodtype['혈액형명'] = Bloodtype['혈액형코드'].str[-2:].map({
    '01': 'O',
    '02': 'A',
    '03': 'B',
    '04': 'AB'
})

Bloodtype[['혈액형코드','혈액형명']]
# RH-,+ NAN 값 제거
Bloodtype = Bloodtype[Bloodtype['혈액형코드'] != 'A00']
# Bloodtype['혈액형명'].unique()

 

In [ ]:
# 결측치확인
Bloodtype['헌혈건수'].isnull().sum()

np.int64(0)

In [ ]:
# 데이터확인
Bloodtype.tail()

,헌혈건수,혈액형명,혈액형코드,단위
23033,26,AB,A0204,건
23034,22,AB,A0204,건
23035,36,AB,A0204,건
23036,36,AB,A0204,건
23037,31,AB,A0204,건


In [ ]:
# 데이터 확인
# 단위 drop
Bloodtype=Bloodtype.drop(columns=['단위'])
Bloodtype.head()
# 저장
Bloodtype.to_csv("Bloodtype_clean.csv", index=False, encoding="utf-8-sig")


In [ ]:
Bloodtype.head()

,헌혈건수,혈액형명,혈액형코드,단위
1098,620195,O,A0101,건
1099,628496,O,A0101,건
1100,571421,O,A0101,건
1101,640707,O,A0101,건
1102,700095,O,A0101,건


# 년도별 헌혈자 통계

In [ ]:
import requests
import pandas as pd

url = "https://kosis.kr/openapi/statisticsData.do?method=getMeta&apiKey=NDM2YzQ2NTdiNTBkM2QyNDg4ZjYyNDYzNzg4OTQwY2U=&format=json&type=TBL&orgId=445&tblId=DT_445001_007&jsonVD=Y"

params = {
    "method": "getList",
    "apiKey": "YOUR_KEY",
    "itmId": "T001+T002+T003+T004+T005+T006+T007+T008",
    "objL1": "A05",   # ← 핵심 (예: 시도/직업/혈액형 등)
    "format": "json",
    "jsonVD": "Y",
    "prdSe": "Y",
    "startPrdDe": "2005",
    "endPrdDe": "2024",
    "orgId": "445",
    "tblId": "DT_445001_007"
}

def get_data(url, params):
    res = requests.get(url, params=params)

    # 1. 상태 체크
    if res.status_code != 200:
        print("HTTP ERROR:", res.status_code)
        print(res.text)
        return pd.DataFrame()

    # 2. JSON 변환 안전 처리
    try:
        data = res.json()
    except Exception:
        print("JSON 아님 → 응답 내용:")
        print(res.text)
        return pd.DataFrame()

    # 3. API 구조 체크
    if isinstance(data, dict) and "list" in data:
        return pd.DataFrame(data["list"])

    return pd.DataFrame(data)

df = get_data(url, params)
df.head()

if "err" in data:
    print("API 오류:", data)
    return pd.DataFrame()

return pd.DataFrame(data.get("list", []))


df = get_data(url, params)

df.head()
df.to_csv("year.csv", index=False, encoding="utf-8-sig")


HTTP ERROR: 404



<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Transitional//EN" "http://www.w3.org/TR/xhtml1/DTD/xhtml1-transitional.dtd">
<html lang="ko">
<head>
	<meta charset="UTF-8">
	<meta http-equiv="X-UA-Compatible" content="IE=Edge">
	<meta name="viewport" content="width=device-width, initial-scale=1">
	<title>KOSIS 공유서비스</title>
	<link rel="stylesheet" href="/openapi/css/error.css">
	
	<script src="/openapi/js/jquery-3.7.0.min.js"></script>
	<script type="text/javascript">
		$(document).ready(function(){
			setTimeout(function() {
				location.href="/openapi/index.jsp";
			}, 5000);
		});
	</script>	
</head>

<body>  

	<div class="error_wrap">
		<div class="center">
			<h1 class="logo">
				<a href="/openapi/index.jsp"><img src="/openapi/image/error/logo.png" alt="KOSIS공유서비스" /></a>
			</h1>
			<h2 class="title">국가통계포털 <span class="text_blue2">사이트 변경안내</span></h2>
			<div class="error_box">
				<p>국가통계포털 서비스를 이용하여 주셔서 감사합니다.</p>
				<p><span class="text_blue">국가통계포털 서비스 개편에

SyntaxError: 'return' outside function (2435111542.py, line 48)

In [74]:
year.head()
# year.columns

#'시점', '총 헌혈실적 (건)', '총 인구 (명)', '헌혈률 (%)', '헌혈가능인구 (16~69세) (명)',
    #    '헌혈가능인구대비 헌혈률 (%)', '헌혈자 실인원 (명)', '헌혈자 1인당 평균 헌혈실적 (건)',
    #    '실제 국민 헌혈률 (%)'

,시점,총 헌혈실적 (건),총 인구 (명),헌혈률 (%),헌혈가능인구 (16~69세) (명),헌혈가능인구대비 헌혈률 (%),헌혈자 실인원 (명),헌혈자 1인당 평균 헌혈실적 (건),실제 국민 헌혈률 (%)
0,2005,2274336,48138077,4.72,33895487,6.71,1544784,1.47,4.56
1,2006,2302542,48371946,4.76,34127502,6.75,1508229,1.53,4.42
2,2007,2087762,48597652,4.30,34356556,6.08,1421906,1.47,4.14
3,2008,2347184,48948698,4.80,34708544,6.76,1526914,1.54,4.40
4,2009,2569954,49182038,5.23,34995947,7.34,1596809,1.61,4.56


In [ ]:
rename_dict = {
    "C1_OBJ_NM": "지역구분",

    "C1": "지역코드",
    "C1_NM": "지역명",

    "DT": "값",

    "PRD_DE": "기준연도",
    "PRD_SE": "주기",

    "ITM_ID": "지표ID",
    "ITM_NM": "지표명",

    "TBL_ID": "테이블ID",
    "TBL_NM": "테이블명",

    "UNIT_NM": "단위",
    "UNIT_NM_ENG": "단위(영문)",

    "ORG_ID": "기관ID",
    "LST_CHN_DE": "수정일"
}

year= year.rename(columns=rename_dict)

In [ ]:
# 필요한 컬럼
year=year[['기준연도','값','단위','지역명']]

# 단위 분리
# % 제거
year = year[year['단위'] != '%'].copy()

year['값'] = pd.to_numeric(year['값'], errors='coerce')

#  pivot (건, 명 만들기)
pivot_df = year.pivot_table(
    index=['기준연도','지역명'],
    columns='단위',
    values='값'
).reset_index()

# #  원본에 붙이기
# result = year.merge(pivot_df, on=['기준연도','지역명'], how='left')

#  단위 컬럼 제거 + 컬럼 정리
years = pivot_df[['기준연도', '건', '명','지역명']]
# 합계 삭제
years = years[years['지역명']!='합계']
years.head()
years.info()

# 기준연도 타입변경
years['기준연도']=years['기준연도'].astype(int)
years['건']=years['건'].astype(int)
years['명']=years['명'].astype(int)



# 저장
years.to_csv("years_data.csv", index=False, encoding="utf-8-sig")

<class 'pandas.core.frame.DataFrame'>
Index: 13 entries, 0 to 12
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   기준연도    13 non-null     object 
 1   건       13 non-null     float64
 2   명       13 non-null     float64
 3   지역명     13 non-null     object 
dtypes: float64(2), object(2)
memory usage: 520.0+ bytes


# 헌혈방법별 헌혈통계

In [ ]:
url='https://kosis.kr/openapi/Param/statisticsParameterData.do?method=getList&apiKey=NDM2YzQ2NTdiNTBkM2QyNDg4ZjYyNDYzNzg4OTQwY2U=&itmId=T001+T002+&objL1=ALL&objL2=ALL&objL3=ALL&objL4=&objL5=&objL6=&objL7=&objL8=&format=json&jsonVD=Y&prdSe=Y&startPrdDe=2005&endPrdDe=2024&orgId=445&tblId=DT_445001_001'

def get_data(url):
    response = requests.get(url)
    response.raise_for_status()
    
    data = response.json()
    
    if isinstance(data, dict):
        data = data.get("list", data)
    
    return pd.DataFrame(data)

                               

method = get_data(url)
method.head()

# 원본저장
method.to_csv("method(2005~2024).csv", index=False, encoding="utf-8-sig")    

rename_dict = {
    "C1_OBJ_NM": "성별구분",
    "C1_OBJ_NM_ENG": "성별구분(영문)",

    "C1": "성별코드",
    "C1_NM": "성별명",
    "C1_NM_ENG": "성별명(영문)",

    "C2_OBJ_NM": "헌혈방법구분",
    "C2": "헌혈방법코드",
    "C2_NM": "헌혈방법명",

    "C3_OBJ_NM": "혈액원구분",
    "C3": "혈액원코드",
    "C3_NM": "혈액원명",

    "DT": "헌혈건수",

    "PRD_DE": "기준연도",
    "PRD_SE": "기간구분",

    "ITM_ID": "항목ID",
    "ITM_NM": "항목명",

    "TBL_ID": "테이블ID",
    "TBL_NM": "테이블명",

    "UNIT_NM": "단위",
    "UNIT_NM_ENG": "단위(영문)",

    "ORG_ID": "기관ID",
    "LST_CHN_DE": "수정일"
}

method = method.rename(columns=rename_dict)
method['헌혈방법명'].unique() #'합계', '전혈헌혈', '320mL', '400mL', '성분헌혈', '혈장', '혈소판', '다종성분', '기타

array(['합계', '전혈헌혈', '320mL', '400mL', '성분헌혈', '혈장', '혈소판', '다종성분', '기타'],
      dtype=object)

In [ ]:
method.head()
method.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16656 entries, 0 to 16655
Data columns (total 22 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   성별구분      16656 non-null  object
 1   헌혈방법명     16656 non-null  object
 2   혈액원코드     16656 non-null  object
 3   헌혈건수      16656 non-null  object
 4   헌혈방법코드    16656 non-null  object
 5   성별코드      16656 non-null  object
 6   기간구분      16656 non-null  object
 7   단위(영문)    16656 non-null  object
 8   항목ID      16656 non-null  object
 9   테이블ID     16656 non-null  object
 10  항목명       16656 non-null  object
 11  테이블명      16656 non-null  object
 12  기준연도      16656 non-null  object
 13  수정일       16656 non-null  object
 14  성별명       16656 non-null  object
 15  단위        16656 non-null  object
 16  혈액원명      16656 non-null  object
 17  혈액원구분     16656 non-null  object
 18  기관ID      16656 non-null  object
 19  성별구분(영문)  16656 non-null  object
 20  헌혈방법구분    16656 non-null  object
 21  성별명(영문)   11

In [ ]:
method =method[['헌혈방법명', '헌혈건수', '헌혈방법코드', '단위']]
method['헌혈방법명'].unique() # 삭제 = 합계,기타,'320mL', '400mL' 
method = method[~method['헌혈방법명'].isin(['합계', '기타', '320ml', '400ml'])]

method['헌혈건수'].unique()

method['헌혈방법코드'].unique() 
#A01', 'A0101', 'A0102', 'A02', 'A0201', 'A0202', 'A0203
method = method[~method['헌혈방법명'].isin(['합계', '기타', '320mL', '400mL'])]
 # % 삭제
method = method[method['단위'] != '%']

method['헌혈방법명'].unique()  #전혈헌혈', '성분헌혈', '혈장', '혈소판', '다종성분'
# method[['헌혈방법명','헌혈방법코드']]

# 단위 삭제
method=method.drop(columns=['단위'])
# 저장
method.to_csv("method_clean.csv", index=False, encoding="utf-8-sig")

## 헌혈방법 두가지 
> 전혈,성분헌혈

| 값        | 의미                  |
| -----    | ------------------- |
| 합계      | 전체 합산               |
| 전혈헌혈  | Whole Blood (전체 혈액) |
| 320mL    | 전혈헌혈 (320ml 기준)     |
| 400mL    | 전혈헌혈 (400ml 기준)     |
| 성분헌혈  | Apheresis (성분헌혈 전체) |
| 혈장      | Plasma donation     |
| 혈소판    | Platelet donation   |
| 다종성분  | 복합 성분헌혈             |
| 기타      | 기타 헌혈방법             |


# 수혈용 혈액 보유 추이

In [ ]:
url ='https://kosis.kr/openapi/Param/statisticsParameterData.do?method=getList&apiKey=NDM2YzQ2NTdiNTBkM2QyNDg4ZjYyNDYzNzg4OTQwY2U=&itmId=T001+&objL1=ALL&objL2=&objL3=&objL4=&objL5=&objL6=&objL7=&objL8=&format=json&jsonVD=Y&prdSe=Y&startPrdDe=2005&endPrdDe=2024&orgId=445&tblId=DT_445001_011'

def get_data(url):
    response = requests.get(url)
    response.raise_for_status()
    
    data = response.json()
    
    if isinstance(data, dict):
        data = data.get("list", data)
    
    return pd.DataFrame(data)


stock = get_data(url)
stock.head()

# 원본저장
stock.to_csv("stock(2005~2024).csv", index=False, encoding="utf-8-sig")    
                            


rename_dict = {
    "C1_OBJ_NM": "구분",

    "C1": "일자코드",
    "C1_NM": "일자",

    "DT": "혈액보유량",

    "PRD_SE": "주기",
    "PRD_DE": "기준연도",

    "ITM_ID": "항목ID",
    "ITM_NM": "항목명",

    "TBL_ID": "테이블ID",
    "TBL_NM": "테이블명",

    "UNIT_NM": "단위",
    "UNIT_NM_ENG": "단위(영문)",

    "ORG_ID": "기관ID",
    "LST_CHN_DE": "수정일"
}

stock = stock.rename(columns=rename_dict)
# 최소 혈액보유량 일자 찾기
stock['혈액보유량']


# 월별데이터로변경

stock['혈액보유량'] = pd.to_numeric(stock['혈액보유량'], errors='coerce')
stock['월'] = stock['일자'].str[:2]

# monthly = stock.groupby("월")["혈액보유량"].mean().reset_index()

In [ ]:
stock.columns
# 구분', '혈액보유량', '일자코드', '주기', '단위(영문)', '항목ID', '테이블ID', '항목명', '테이블명',
    #    '기준연도', '수정일', '일자', '단위', '기관ID', '월'
# 컬럼 삭제
stock=stock[['혈액보유량','월']]
stock=stock.sort_values(by='월',ascending=True)

stock['월'] = stock['월'].astype(int)

stock.info()

# 원본저장
stock.to_csv("stock_clean.csv", index=False, encoding="utf-8-sig")    
                            

<class 'pandas.core.frame.DataFrame'>
Index: 7308 entries, 0 to 5744
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   혈액보유량   7306 non-null   float64
 1   월       7308 non-null   int64  
dtypes: float64(1), int64(1)
memory usage: 171.3 KB


- data 월별헌혈건수
- total 총헌혈수(지역별)
- age 연령대별
- job 직업별 
- location 위치별
- Bloodtype 혈액형별
- year_per (년도별 헌혈자 통계)
- method 헌혈방법별 (성분,전혈)
- stock 혈액재고(일별)

> 원본데이터 저장



In [1]:
import pandas as pd
import os

base = r"c:\AI project\전처리"

files = [
    "age_clean.csv",
    "Bloodtype_clean.csv",
    "job_clean.csv",
    "location_clean.csv",
    "method_clean.csv",
    "month_clean.csv",
    "region_clean.csv",
    "stock_clean.csv",
    'year_clean.csv'
]

BD = {}

for file in files:
    path = os.path.join(base, file)
    
    print("loading:", path)
    
    df = pd.read_csv(path)
    
    df = df.rename(columns={'건': '헌혈건수'})
    
    BD[file] = df





age= BD["age_clean.csv"]
Bloodtype= BD["Bloodtype_clean.csv"]
job= BD["job_clean.csv"]
location= BD["location_clean.csv"]
method= BD["method_clean.csv"]
month= BD["month_clean.csv"]
region= BD["region_clean.csv"]
stock= BD["stock_clean.csv"]
year= BD['year_clean.csv']

loading: c:\AI project\전처리\age_clean.csv
loading: c:\AI project\전처리\Bloodtype_clean.csv
loading: c:\AI project\전처리\job_clean.csv
loading: c:\AI project\전처리\location_clean.csv
loading: c:\AI project\전처리\method_clean.csv
loading: c:\AI project\전처리\month_clean.csv
loading: c:\AI project\전처리\region_clean.csv
loading: c:\AI project\전처리\stock_clean.csv
loading: c:\AI project\전처리\year_clean.csv


In [64]:
# # =========================================
# # 데이터 타입 변경,라벨 인코딩
# # =========================================
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder

# ========================================
# age
# =======================================
# age= 연령대삭제
# age = age.drop(columns=['연령대'])


# ========================================
# job
# =======================================

# job = 직업명 인코딩(int),직업코드삭제

# job = job.drop(columns=['직업코드'])


le = LabelEncoder()
job["직업명"] = le.fit_transform(job["직업명"])
job.info()

# ========================================
# location
# =======================================

# location = 장소코드삭제,장소명(인코딩),추정헌혈유형(삭제),혈액원명 삭제

location=location[['장소명','헌혈건수']]

location['장소명'] = le.fit_transform(location['장소명'])
location.info()


# ========================================
# region
# =======================================
# region =시도명(인코딩)
region['시도명']= le.fit_transform(region['시도명'])

# ========================================
# method
# =======================================
# method=헌혈방법코드 삭제 ,헌혈방법명 인코딩,헌혈건수 int
# method = method.drop(columns=['헌혈방법코드'])
method['헌혈방법명'] = le.fit_transform(method['헌혈방법명'])
# method = method.drop(columns=['헌혈방법코드'])
method.info()

# ========================================
# bloodtype
# =======================================
# bloodtype = 혈액형명인코딩,혈액형코드삭제,헌혈건수int
# Bloodtype['헐액형명'] = le.fit_transform(Bloodtype['혈액형명'])
# Bloodtype = Bloodtype.drop(columns=['혈액형코드'])
# Bloodtype['헌혈건수']=Bloodtype['헌혈건수'].astype(int)

Bloodtype[Bloodtype['헌혈건수']=='-']

# #9723	-	AB	1
# #9725	-	AB	1

import numpy as np


Bloodtype["헌혈건수"] = Bloodtype["헌혈건수"].replace("-", np.nan)
Bloodtype["헌혈건수"] = pd.to_numeric(Bloodtype["헌혈건수"])

Bloodtype.isnull().sum() # 결측치 2개

# 결측치 ab형 평균건수로 대체
# ab_mean = Bloodtype[Bloodtype["혈액형명"] == "AB"]["헌혈건수"].mean()

# Bloodtype.loc[
#     (Bloodtype["혈액형명"] == "AB") &
#     (Bloodtype["헌혈건수"].isnull()),
#     "헌혈건수"
# ] = ab_mean

# Bloodtype.isnull().sum() 
# Bloodtype.head()

Bloodtype=Bloodtype.drop(columns=['혈액형코드'])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 160 entries, 0 to 159
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   직업명     160 non-null    int64
 1   기준연도    160 non-null    int64
 2   헌혈건수    160 non-null    int64
dtypes: int64(3)
memory usage: 3.9 KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6846 entries, 0 to 6845
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   장소명     6846 non-null   int64
 1   헌혈건수    6846 non-null   int64
dtypes: int64(2)
memory usage: 107.1 KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5045 entries, 0 to 5044
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   헌혈방법명   5045 non-null   int64 
 1   헌혈건수    5045 non-null   object
dtypes: int64(1), object(1)
memory usage: 79.0+ KB


In [69]:
BD2 = {
    "age": age,
    "job": job,
    "location": location,
    "region": region,
    "method": method,
    "Bloodtype": Bloodtype,
    "stock": stock
}


import os

# 저장 폴더 생성
os.makedirs('전처리2', exist_ok=True)

# 데이터 저장
for name, df in BD2.items():
    
    save_path = f'전처리2/{name}2.csv'
    
    df.to_csv(save_path,
              index=False,
              encoding='utf-8-sig')
    
    print(f'{save_path} 저장 완료')

전처리2/age2.csv 저장 완료
전처리2/job2.csv 저장 완료
전처리2/location2.csv 저장 완료
전처리2/region2.csv 저장 완료
전처리2/method2.csv 저장 완료
전처리2/Bloodtype2.csv 저장 완료
전처리2/stock2.csv 저장 완료
